# Copula-GARCH

In [21]:
import numpy as np
import pandas as pd

from arch import arch_model 

import yfinance as yf

## Functions

Fitting GARCH models

In [ ]:
def fit_garch(returns):
    """
    Fit a ARMA(1,1)-GARCH(1,1) model to the returns and return the fitted model.
    """
    model = arch_model(returns*100, vol="Garch", p=1, q=1, dist="t") # scale returns for better convergence

    res = model.fit(disp="off", show_warning=False) # disp="off" to suppress output, show_warning=False to ignore convergence warnings

    return res

## 1. Basics

### 1.1 Load data

In [22]:
TICKERS = ["NVDA", "AAPL", "WMT", "LLY", "JPM"]

prices = yf.download(TICKERS, start="2020-01-01", end="2024-06-30")["Close"]

returns = np.log(prices / prices.shift(1)).dropna() # log returns

returns.head()

[*********************100%***********************]  5 of 5 completed


Ticker,AAPL,JPM,LLY,NVDA,WMT
Date,,,,,
2020-01-03,-0.009770,-0.013284,-0.003334,-0.016135,-0.008867
2020-01-06,0.007937,-0.000796,0.003712,0.004185,-0.002038
2020-01-07,-0.004714,-0.017147,0.001888,0.012034,-0.009308
2020-01-08,0.015958,0.007771,0.009016,0.001874,-0.003438
2020-01-09,0.021019,0.003645,0.016393,0.010923,0.010278


### 1.2 Setup

In [23]:
ROLL_WIND = 250
FORECAST = 100
N_NIM = 10000

ALPHA = 0.05

SEED = 42

weights = np.ones(len(TICKERS)) / len(TICKERS)

## 2. Rolling window

## SANDBOX 

### Load return data

In [24]:
returns_aapl = returns["AAPL"]
returns_aapl.head()

Date
2020-01-03   -0.009770
2020-01-06    0.007937
2020-01-07   -0.004714
2020-01-08    0.015958
2020-01-09    0.021019
Name: AAPL, dtype: float64

### Using R's rugarch package in python

In [25]:
# rpy2 
import rpy2

import rpy2.robjects as ro
from rpy2.robjects.packages import importr # importr is used to import R packages


In [5]:
R --version

NameError: name 'R' is not defined

In [26]:
# loading R packages
utils = importr('utils') # utils is used to install R packages

copula = importr("copula")
rugarch = importr("rugarch")


In [27]:
# check versions of R packages
rugarch.__version__

'1.5-5'

In [28]:
# check versions of R packages
copula.__version__

'1.1-7'

Fitting the GARCH model

In [ ]:
# GARCH(1,1) specification in R
spec = rugarch.ugarchspec(variance_model = ro.ListVector({'model': "sGARCH", 'garchOrder': ro.IntVector([1, 1])}), # specify GARCH(1,1) model
                          mean_model = ro.ListVector({'armaOrder': ro.IntVector([1, 1]), 'include.mean': True}), # specify ARMA(1,1) model for the mean
                          distribution_model = "std") # specify Student's t distribution for the innovations

In [ ]:
# fit the model to the returns (scale returns for better convergence)
fit = rugarch.ugarchfit(spec, ro.FloatVector(returns_aapl.values * 100)) # fit the model to the returns (scale returns for better convergence)

Extracting standardized residuals

In [ ]:
residuals = np.array(rugarch.residuals(fit)) # Get the residuals from the fitted model

sigma = np.array(rugarch.sigma(fit)) # Get the conditional volatility from the fitted model

residuals_standardized = residuals / sigma # Standardize the residuals

array([[-0.53604618],
       [ 0.32410836],
       [-0.30941715],
       ...,
       [ 0.95811584],
       [ 0.16134166],
       [-0.93761364]], shape=(1129, 1))

Forecast values

In [31]:
# One-step-ahead forecast for mean and conditional volatility
forecast = rugarch.ugarchforecast(fit, n_ahead=1)

In [ ]:
# Mean forecast for the next day (scale back the mean forecast)
mu_forecast = np.array(rugarch.fitted(forecast))[0] / 100 # scale back the mean forecast

# Volatility forecast for the next day (scale back the volatility forecast)
sigma_forecast = np.array(rugarch.sigma(forecast))[0] / 100 # scale back the volatility forecast

In [44]:
mu_forecast, sigma_forecast

(array([0.00147197]), array([0.01847441]))